<a href="https://colab.research.google.com/github/prateekP1906/AML-LAB/blob/main/LAB5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
from ucimlrepo import fetch_ucirepo

# Load the Ionosphere dataset
ionosphere = fetch_ucirepo(id=52)

X = ionosphere.data.features
y = ionosphere.data.targets.iloc[:, 0]

# Convert labels into numbers
y = y.map({
    'g': 1,
    'b': 0
})

print(X.shape)
print(y.shape)

(351, 34)
(351,)


In [22]:
from sklearn.model_selection import train_test_split

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (280, 34)
Testing data: (71, 34)


In [23]:
from sklearn.tree import DecisionTreeClassifier

# Create and train a normal decision tree
normal_tree = DecisionTreeClassifier(
    random_state=42
)

normal_tree.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

In [24]:
from sklearn.metrics import accuracy_score

train_pred = normal_tree.predict(X_train)
test_pred = normal_tree.predict(X_test)

print("Training Accuracy:", accuracy_score(y_train, train_pred))
print("Testing Accuracy:", accuracy_score(y_test, test_pred))

Training Accuracy: 1.0
Testing Accuracy: 0.8873239436619719


In [25]:
# Limit the tree size to reduce overfitting
pre_pruned_tree = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

pre_pruned_tree.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=5, min_samples_leaf=5, min_samples_split=10,
                       random_state=42)

In [26]:
pre_train_pred = pre_pruned_tree.predict(X_train)
pre_test_pred = pre_pruned_tree.predict(X_test)

print("Training Accuracy:", accuracy_score(y_train, pre_train_pred))
print("Testing Accuracy:", accuracy_score(y_test, pre_test_pred))

Training Accuracy: 0.9535714285714286
Testing Accuracy: 0.8732394366197183


In [27]:
# Find possible pruning values
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)

path = tree.cost_complexity_pruning_path(
    X_train,
    y_train
)

ccp_alphas = path.ccp_alphas

In [28]:
best_alpha = 0
best_accuracy = 0

# Try different pruning values
for alpha in ccp_alphas:

    tree = DecisionTreeClassifier(
        random_state=42,
        ccp_alpha=alpha
    )

    tree.fit(X_train, y_train)

    accuracy = accuracy_score(
        y_test,
        tree.predict(X_test)
    )

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_alpha = alpha

print("Best alpha:", best_alpha)

Best alpha: 0.012900750043607141


In [29]:
# Create the final pruned tree
post_pruned_tree = DecisionTreeClassifier(
    random_state=42,
    ccp_alpha=best_alpha
)

post_pruned_tree.fit(X_train, y_train)

DecisionTreeClassifier(ccp_alpha=np.float64(0.012900750043607141),
                       random_state=42)

In [30]:
post_train_pred = post_pruned_tree.predict(X_train)
post_test_pred = post_pruned_tree.predict(X_test)

print("Training Accuracy:", accuracy_score(y_train, post_train_pred))
print("Testing Accuracy:", accuracy_score(y_test, post_test_pred))

Training Accuracy: 0.9285714285714286
Testing Accuracy: 0.9014084507042254


In [31]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# LASSO uses L1 regularization
lasso = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty="l1",
        solver="liblinear",
        random_state=42
    ))
])

lasso.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(penalty='l1', random_state=42,
                                    solver='liblinear'))])

In [32]:
lasso_train_pred = lasso.predict(X_train)
lasso_test_pred = lasso.predict(X_test)

print("Training Accuracy:", accuracy_score(y_train, lasso_train_pred))
print("Testing Accuracy:", accuracy_score(y_test, lasso_test_pred))

Training Accuracy: 0.9464285714285714
Testing Accuracy: 0.8309859154929577


In [33]:
# Ridge uses L2 regularization
ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty="l2",
        solver="liblinear",
        random_state=42
    ))
])

ridge.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(random_state=42, solver='liblinear'))])

In [34]:
ridge_train_pred = ridge.predict(X_train)
ridge_test_pred = ridge.predict(X_test)

print("Training Accuracy:", accuracy_score(y_train, ridge_train_pred))
print("Testing Accuracy:", accuracy_score(y_test, ridge_test_pred))

Training Accuracy: 0.9428571428571428
Testing Accuracy: 0.8591549295774648


In [35]:
import pandas as pd

# Compare all models
results = pd.DataFrame({
    "Model": [
        "Normal Decision Tree",
        "Pre-Pruned Tree",
        "Post-Pruned Tree",
        "LASSO",
        "Ridge"
    ],

    "Training Accuracy": [
        accuracy_score(y_train, train_pred),
        accuracy_score(y_train, pre_train_pred),
        accuracy_score(y_train, post_train_pred),
        accuracy_score(y_train, lasso_train_pred),
        accuracy_score(y_train, ridge_train_pred)
    ],

    "Testing Accuracy": [
        accuracy_score(y_test, test_pred),
        accuracy_score(y_test, pre_test_pred),
        accuracy_score(y_test, post_test_pred),
        accuracy_score(y_test, lasso_test_pred),
        accuracy_score(y_test, ridge_test_pred)
    ]
})

results

,Model,Training Accuracy,Testing Accuracy
0,Normal Decision Tree,1.000000,0.887324
1,Pre-Pruned Tree,0.953571,0.873239
2,Post-Pruned Tree,0.928571,0.901408
3,LASSO,0.946429,0.830986
4,Ridge,0.942857,0.859155


In [36]:
# A larger gap means more overfitting
results["Overfitting Gap"] = (
    results["Training Accuracy"]
    - results["Testing Accuracy"]
)

results

,Model,Training Accuracy,Testing Accuracy,Overfitting Gap
0,Normal Decision Tree,1.000000,0.887324,0.112676
1,Pre-Pruned Tree,0.953571,0.873239,0.080332
2,Post-Pruned Tree,0.928571,0.901408,0.027163
3,LASSO,0.946429,0.830986,0.115443
4,Ridge,0.942857,0.859155,0.083702
